In [2]:
import python.Utils.FilePaths

ModuleNotFoundError: No module named 'python'

In [63]:
import requests
import pandas as pd
import sqlite3
import datetime
from os import path

In [64]:
#--------Directories ---------------------------------------
## Project directory
project_dir = path.dirname(path.dirname(path.abspath('')))

## Location of database files
metadata_dir = path.join(project_dir,"files", "data", "lfs", "UWB", "metadata")

In [65]:
def api_request(url, timeout_seconds=60):
    """Fetches API data with a defined timeout handling."""
    try:
        r = requests.get(url, timeout=timeout_seconds)
        if r.status_code == 200:
            return r.json()
        print(f"Error with API request, code: {r.status_code}")
    except requests.exceptions.Timeout:
        print(f"API request timed out after {timeout_seconds} seconds.")
    except requests.exceptions.RequestException as e:
        print(f"API request faile^d: {e}")
    return None

In [66]:
## Download all measurement station metadata from UWB-APi
url = 'https://luftdaten.umweltbundesamt.de/api/air-data/v4/stations/json?use=airquality&lang=en'
r_station = api_request(url=url, timeout_seconds=10)

In [67]:
#-------Create stations pandas dataframe 

## The column names are defined by the apis 'indices'
station_indices = r_station['indices']
station_indices = [i.replace(" ","_") for i in station_indices]

df_stations = pd.DataFrame([], columns=station_indices)

station_json_data = list(r['data'].items())
for i, item  in enumerate(station_json_data):
    df_stations.loc[i] = item[:][:][1][:]

df_stations =df_stations.set_index(keys="station_id")


## Save stations metadata as csv
file_name= "UWB_stations_metadata.csv"
file_path = path.join(metadata_dir, file_name)
df_stations.to_csv(file_path)


In [68]:
## Download all measurement station metadata from UWB-APi
url_comp = "https://luftdaten.umweltbundesamt.de/api/air-data/v4/components/json?lang=de"
r_comp = api_request(url=url_comp, timeout_seconds=10)

In [71]:
#-------Create components pandas dataframe 

## The column names are defined by the apis 'indices'
comp_indices = r_comp['indices']
comp_indices = [i.replace(" ","_") for i in comp_indices]

df_components = pd.DataFrame([], columns=comp_indices)

for i, comp in enumerate(range(1,13)):
    df_components.loc[i] = r_comp[str(comp)]

df_components =df_components.set_index(keys="component_id")

## Save components metadata as csv
file_name= "UWB_components_metadata.csv"
file_path = path.join(metadata_dir, file_name)
df_components.to_csv(file_path)